In [9]:
!pip uninstall -y -q bitsandbytes 2>/dev/null
!pip install -q -U transformers==4.46.0 datasets==3.0.1 peft==0.13.2 \
    accelerate==1.0.1 trl==0.11.4 evaluate scikit-learn


In [10]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # silence a harmless fork warning

import torch, random, json
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import numpy as np
import inspect

random.seed(42)
torch.manual_seed(42)


CUDA available: True
Device: Tesla T4
VRAM: 15.6 GB


##  Load the base model 



In [11]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("Loaded. Parameter count:", sum(p.numel() for p in base_model.parameters()) / 1e6, "M")


Loaded. Parameter count: 1543.714304 M


In [12]:
def chat(model, user_msg, max_new_tokens=60, system_msg=None):
    messages = []
    if system_msg:
        messages.append({"role": "system", "content": system_msg})
    messages.append({"role": "user", "content": user_msg})
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()

print(chat(base_model, "Say hello in one short sentence."))


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Hello! How can I assist you today?


## Dataset

small, hand-curated synthetic dataset of support tickets (~200 examples,
50 per category).


Categories: Billing, Technical, Account, General.


In [13]:

billing_seeds = [
    "I was charged twice for my subscription this month.",
    "Can I get a refund for the annual plan I just bought?",
    "My invoice shows the wrong amount, please correct it.",
    "How do I update my credit card on file?",
    "Why did my monthly price go up without notice?",
    "I need a copy of last year's invoices for tax purposes.",
    "The discount code I used didn't apply at checkout.",
    "I want to downgrade my plan to save money.",
    "There's a charge on my statement I don't recognize.",
    "Can you cancel my subscription and refund the last payment?",
]
technical_seeds = [
    "The app crashes every time I try to upload a file.",
    "I'm getting a 500 error when I try to log in.",
    "The mobile app won't sync with the desktop version.",
    "Push notifications stopped working after the last update.",
    "The export button does nothing when I click it.",
    "My dashboard is showing a blank white screen.",
    "The API is returning a 429 rate limit error constantly.",
    "Video calls keep freezing after about two minutes.",
    "Dark mode is broken on the settings page.",
    "The search feature returns no results even for common terms.",
]
account_seeds = [
    "I forgot my password and the reset email never arrives.",
    "How do I change the email address linked to my account?",
    "I can't log in even though my password is correct.",
    "Please delete my account and all associated data.",
    "I need to transfer ownership of this account to a colleague.",
    "Two-factor authentication is locked and I can't get in.",
    "How do I merge two accounts I accidentally created?",
    "My account was suspended and I don't know why.",
    "I want to add a team member to my workspace.",
    "Can you change my username?",
]
general_seeds = [
    "Do you offer discounts for students?",
    "What are your customer support hours?",
    "Is there a mobile app for iOS?",
    "Can I get a demo before subscribing?",
    "Do you have an affiliate program?",
    "What integrations does your product support?",
    "Is my data GDPR compliant if I'm based in the EU?",
    "How can I suggest a new feature?",
    "Where can I find your API documentation?",
    "Do you have a roadmap I can see publicly?",
]

# Light templated variations to reach ~50/category while staying clean & controlled.
prefixes = ["", "Hi, ", "Hello, ", "Hey team, ", "Quick question: ", "Urgent: ", ""]

def expand(seeds, label, target=50):
    out = []
    i = 0
    while len(out) < target:
        seed = seeds[i % len(seeds)]
        prefix = prefixes[i % len(prefixes)]
        text = (prefix + seed[0].lower() + seed[1:]) if prefix else seed
        out.append({"text": text, "label": label})
        i += 1
    return out

data = (
    expand(billing_seeds, "Billing")
    + expand(technical_seeds, "Technical")
    + expand(account_seeds, "Account")
    + expand(general_seeds, "General")
)
random.shuffle(data)
print("Total examples:", len(data))
print(data[0])


Total examples: 200
{'text': 'Hello, the API is returning a 429 rate limit error constantly.', 'label': 'Technical'}


## Train/test split 

In [14]:

split_idx = int(len(data) * 0.85)
train_data = data[:split_idx]
test_data = data[split_idx:]
print(f"Train: {len(train_data)}  Test: {len(test_data)}")

SYSTEM_PROMPT = (
    "You are a support ticket triage assistant. Classify the ticket into exactly one "
    "category: Billing, Technical, Account, or General. Respond with only the category name."
)

def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["text"]},
        {"role": "assistant", "content": example["label"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = Dataset.from_list(train_data).map(format_example)
print(train_ds[0]["text"])


Train: 170  Test: 30


Map:   0%|          | 0/170 [00:00<?, ? examples/s]

<|im_start|>system
You are a support ticket triage assistant. Classify the ticket into exactly one category: Billing, Technical, Account, or General. Respond with only the category name.<|im_end|>
<|im_start|>user
Hello, the API is returning a 429 rate limit error constantly.<|im_end|>
<|im_start|>assistant
Technical<|im_end|>



## Before: true baseline predictions 



In [15]:

def predict_no_hint(m, ticket_text):
    return chat(m, f"Categorize this support ticket in one word: {ticket_text}", max_new_tokens=15)

before_preds = []
for ex in test_data:
    pred = predict_no_hint(base_model, ex["text"])  # untouched base model, no LoRA attached yet
    before_preds.append(pred)

for ex, pred in list(zip(test_data, before_preds))[:8]:
    print(f"[true={ex['label']:9}] pred={pred!r:30} | {ex['text']}")


[true=Account  ] pred='Account'                      | I need to transfer ownership of this account to a colleague.
[true=Technical] pred='RateLimit'                    | The API is returning a 429 rate limit error constantly.
[true=Account  ] pred='Suspension'                   | My account was suspended and I don't know why.
[true=General  ] pred='"Hours"'                      | What are your customer support hours?
[true=General  ] pred='GDPR'                         | Hello, is my data GDPR compliant if I'm based in the EU?
[true=Technical] pred='Bug'                          | The app crashes every time I try to upload a file.
[true=Account  ] pred='Delete'                       | Hi, please delete my account and all associated data.
[true=General  ] pred='Support Ticket Categorization Word: Feature Request' | Urgent: how can I suggest a new feature?


##  Set up LoRA and train


In [17]:
import peft.tuners.lora.model as _lora_model_module
_lora_model_module.is_bnb_available = lambda: False
_lora_model_module.is_bnb_4bit_available = lambda: False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


training_args = SFTConfig(
    output_dir="./lora-ticket-classifier",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=256,
)


sft_params = inspect.signature(SFTTrainer.__init__).parameters
tokenizer_kwarg = "processing_class" if "processing_class" in sft_params else "tokenizer"

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    tokenizer_kwarg: tokenizer,
}
if "dataset_text_field" in sft_params:
    trainer_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sft_params:
    trainer_kwargs["max_seq_length"] = 256

trainer = SFTTrainer(**trainer_kwargs)

trainer.train()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/170 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


Step,Training Loss
10,2.613300
20,0.879700
30,0.459900
40,0.362800
50,0.322800


TrainOutput(global_step=55, training_loss=0.8719317284497348, metrics={'train_runtime': 26.8106, 'train_samples_per_second': 31.704, 'train_steps_per_second': 2.051, 'total_flos': 443174431709184.0, 'train_loss': 0.8719317284497348, 'epoch': 5.0})

##  After: fine-tuned model predictions on the same held-out tickets


In [18]:

model.eval()
after_preds = []
for ex in test_data:
    pred = predict_no_hint(model, ex["text"])
    after_preds.append(pred)

correct_before = sum(1 for ex, p in zip(test_data, before_preds) if ex["label"].lower() in p.lower())
correct_after = sum(1 for ex, p in zip(test_data, after_preds) if ex["label"].lower() in p.lower())

print(f"Baseline accuracy (no schema given): {correct_before}/{len(test_data)}")
print(f"Fine-tuned accuracy (no schema given): {correct_after}/{len(test_data)}")


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Baseline accuracy (no schema given): 6/30
Fine-tuned accuracy (no schema given): 8/30


 
**Before/After: 4 concrete examples**



In [19]:

print("="*90)
for ex, before, after in list(zip(test_data, before_preds, after_preds))[:4]:
    print(f"TICKET:   {ex['text']}")
    print(f"TRUE LABEL: {ex['label']}")
    print(f"BEFORE (base model):       {before!r}")
    print(f"AFTER  (LoRA fine-tuned):  {after!r}")
    print("-"*90)


TICKET:   I need to transfer ownership of this account to a colleague.
TRUE LABEL: Account
BEFORE (base model):       'Account'
AFTER  (LoRA fine-tuned):  'Account'
------------------------------------------------------------------------------------------
TICKET:   The API is returning a 429 rate limit error constantly.
TRUE LABEL: Technical
BEFORE (base model):       'RateLimit'
AFTER  (LoRA fine-tuned):  'Rate'
------------------------------------------------------------------------------------------
TICKET:   My account was suspended and I don't know why.
TRUE LABEL: Account
BEFORE (base model):       'Suspension'
AFTER  (LoRA fine-tuned):  'Technical'
------------------------------------------------------------------------------------------
TICKET:   What are your customer support hours?
TRUE LABEL: General
BEFORE (base model):       '"Hours"'
AFTER  (LoRA fine-tuned):  'Hours'
------------------------------------------------------------------------------------------


In [20]:
def predict_matched(m, ticket_text):
    return chat(m, ticket_text, max_new_tokens=10, system_msg=SYSTEM_PROMPT)

before_matched = []
with model.disable_adapter():
    for ex in test_data:
        before_matched.append(predict_matched(model, ex["text"]))

after_matched = []
for ex in test_data:
    after_matched.append(predict_matched(model, ex["text"]))

correct_before_matched = sum(1 for ex, p in zip(test_data, before_matched) if ex["label"].lower() in p.lower())
correct_after_matched = sum(1 for ex, p in zip(test_data, after_matched) if ex["label"].lower() in p.lower())

print(f"Baseline accuracy (matched/training prompt): {correct_before_matched}/{len(test_data)}")
print(f"Fine-tuned accuracy (matched/training prompt): {correct_after_matched}/{len(test_data)}")
print()
print("="*90)
for ex, before, after in list(zip(test_data, before_matched, after_matched))[:4]:
    print(f"TICKET:   {ex['text']}")
    print(f"TRUE LABEL: {ex['label']}")
    print(f"BEFORE (base model, matched prompt):      {before!r}")
    print(f"AFTER  (LoRA fine-tuned, matched prompt):  {after!r}")
    print("-"*90)

Baseline accuracy (matched/training prompt): 18/30
Fine-tuned accuracy (matched/training prompt): 30/30

TICKET:   I need to transfer ownership of this account to a colleague.
TRUE LABEL: Account
BEFORE (base model, matched prompt):      'Account'
AFTER  (LoRA fine-tuned, matched prompt):  'Account'
------------------------------------------------------------------------------------------
TICKET:   The API is returning a 429 rate limit error constantly.
TRUE LABEL: Technical
BEFORE (base model, matched prompt):      'Technical'
AFTER  (LoRA fine-tuned, matched prompt):  'Technical'
------------------------------------------------------------------------------------------
TICKET:   My account was suspended and I don't know why.
TRUE LABEL: Account
BEFORE (base model, matched prompt):      'Account'
AFTER  (LoRA fine-tuned, matched prompt):  'Account'
------------------------------------------------------------------------------------------
TICKET:   What are your customer support hours?

##  Save the LoRA adapter

In [21]:

SAVE_DIR = "/kaggle/working/lora-ticket-classifier-adapter"  # persisted in Kaggle's Output tab
import os
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved adapter to {SAVE_DIR}")
print("To reload later: AutoModelForCausalLM.from_pretrained(base) then PeftModel.from_pretrained(base, path)")


Saved adapter to /kaggle/working/lora-ticket-classifier-adapter
To reload later: AutoModelForCausalLM.from_pretrained(base) then PeftModel.from_pretrained(base, path)


# **Summary:**

**Task:** support ticket classification into 4 categories - Billing, Technical, Account, General.

**Model:** Qwen2.5-1.5B-Instruct (Apache-2.0), fine-tuned with full LoRA in fp16 (rank 16, alpha 32, targeting the attention Q/K/V/O projections — ~4.4M trainable params, 0.28% of the 1.5B total). Trained for 5 epochs on free Kaggle T4 GPU. 


**Data:** a ~200-example hand-curated synthetic dataset (50 per category, 85/15 train/test split).

**Evaluation design:** I tested the base and fine-tuned models under two conditions to separate two different questions:

1. **Matched prompt** (same system prompt used during training, listing the 4 categories) — tests whether fine-tuning improved the model's reliability on the task it was actually trained on.
2. **No-hint prompt** (category list withheld) — a stricter test of whether the learned behavior generalizes to prompt formats the model never saw during training.

**Results:**
- Matched prompt: baseline accuracy 18/30 (60%) → fine-tuned accuracy 30/30 (100%) on held-out test tickets.
- No-hint prompt: gains were more mixed — some outputs showed cleaner formatting (e.g., dropping stray quotes around the label) after fine-tuning, but didn't always converge on the exact trained label text.

**Interpretation:** LoRA fine-tuning produced a clear, measurable improvement (60%→100%) on the task and prompt format it was trained on, showing the fine-tune successfully taught the model to consistently apply our exact category schema instead of inventing its own labels (e.g., base model said "Suspension" or "RateLimit" instead of the schema's "Account"/"Technical"). Generalization to a stricter, unseen prompt format was weaker, which is expected for a small dataset (170 training examples) and a narrow LoRA adapter, it indicates the fine-tune learned the task well within-distribution, with room to improve out-of-distribution robustness with more data or more diverse prompt formats during training.

